In [ ]:
#| hide
from solvedrive.core import *

# solvedrive

> Google Drive client built on the Drive API

solvedrive is an async Python client for Google Drive. It wraps the Drive REST API in a few small classes (`GDrive`, `Drive`, `File`, `Folder`, and their collections), so you can upload and download files, search across My Drive and shared drives, export Google Docs, organize with folders, and share, all with plain method calls.

It is built on [fastgws](https://github.com/AnswerDotAI/fastgws) for OAuth and API access, and runs every call asynchronously, so you `await` each one. Searches come back with the useful fields (`name`, `mimeType`, `size`, `parents`, `trashed`, `webViewLink`) already populated in one request, and reprs render file names as links that open the file in Drive.

## Installation

Install latest from [pypi][pypi]

```sh
$ pip install solvedrive
```

[pypi]: https://pypi.org/project/solvedrive/

## How to use

solvedrive signs in with OAuth 2.0 through [fastgws](https://github.com/AnswerDotAI/fastgws), which expects an OAuth client secret at `~/.config/fastgws/credentials.json`. The fastgws docs cover how to create that file from a Google Cloud project. The first connection opens a browser to authorize the scopes you asked for, then caches the token so later runs don't prompt again.

Create a `GDrive` client with the scopes you need: `readonly` to search and download, `file` to also manage the files your app created, or `full` for everything. `about()` returns the signed-in account, with the address on its `email` attribute.

In [ ]:
gd = await GDrive.init(scopes='full')
about = await gd.about()
about.email

'nc@answer.ai'

### Uploading

`upload` sends bytes or a local path to Drive, returning the new `File`. Its repr shows the essentials, with the name linking to the file in Drive.

In [ ]:
f = await gd.upload(data=b'hello from solvedrive!', name='solvedrive_hello.txt')
f

File(1X0jjggHSWBTQbxaLxtsSNqcDW9uIJpQh: <a href="https://drive.google.com/file/d/1X0jjggHSWBTQbxaLxtsSNqcDW9uIJpQh/view?usp=drivesdk" target="_blank">solvedrive_hello.txt</a> (text/plain, 22B) 2026-07-28T21:08:32.734Z)

### Searching

`search_files` takes Drive's own [query syntax](https://developers.google.com/workspace/drive/api/guides/search-files), the same operators the Drive search box uses, and searches every drive you can see: My Drive, files shared with you, and all shared drives. Results are a `Files` collection with a table repr.

In [ ]:
fs = await gd.search_files("name = 'solvedrive_hello.txt' and trashed=false")
fs

| id | name | type | size | modified |
|-|-|-|-|-|
| 1X0jjggHSWBTQbxaLxtsSNqcDW9uIJpQh | <a href="https://drive.google.com/file/d/1X0jjggHSWBTQbxaLxtsSNqcDW9uIJpQh/view?usp=drivesdk" target="_blank">solvedrive_hello.txt</a> | text/plain | 22B | 2026-07-28 |

### Downloading

`download` returns a file's bytes. Google-native files (Docs, Sheets, Slides) have no bytes of their own, so they are exported to a regular format instead: docx for Docs, xlsx for Sheets, pptx for Slides, or any supported `mime=` you ask for.

In [ ]:
await f.download()

b'hello from solvedrive!'

### Folders

A folder is a file with a special `mimeType`. `create_folder` makes one, `move` puts files in it, and `ls` lists what it holds.

In [ ]:
folder = await gd.create_folder('solvedrive demo')
await f.move(folder)
await folder.ls()

| id | name | type | size | modified |
|-|-|-|-|-|
| 1X0jjggHSWBTQbxaLxtsSNqcDW9uIJpQh | <a href="https://drive.google.com/file/d/1X0jjggHSWBTQbxaLxtsSNqcDW9uIJpQh/view?usp=drivesdk" target="_blank">solvedrive_hello.txt</a> | text/plain | 22B | 2026-07-28 |

### Drives

Every file lives in a drive: your My Drive, or a shared drive owned by an organization. `list_drives` returns them as a `Drives` table, and each `Drive` knows your role on it and scopes `search_files` to its contents. `drive.root` is an ordinary `Folder`, so uploading into and listing a shared drive works like any folder, and every file method sends `supportsAllDrives`, so a `File` behaves the same wherever it lives.

In [ ]:
ds = await gd.list_drives()
ds[0]

Drive(root: <a href="https://drive.google.com/drive/my-drive" target="_blank">My Drive</a>, owner)

### Sharing

`share` grants access, either to an email address or to anyone with the link when none is given. `permissions` lists the current grants, and `unshare` revokes one.

In [ ]:
p = await f.share()
await f.permissions

[{'id': 'anyoneWithLink', 'type': 'anyone', 'role': 'reader'}, {'id': '12635610242443046038', 'type': 'user', 'emailAddress': 'nc@answer.ai', 'role': 'owner'}]

There's more in `solvedrive.core`: working inside shared drives, renaming and copying, trash versus permanent deletion, and operating on a whole `Files` collection at once through Google's batch protocol. See the [docs](https://answerDotAI.github.io/solvedrive/) for the full tour.

### Use as an LLM skill

solvedrive registers a [pyskills](https://github.com/AnswerDotAI/pyskills) skill, so an LLM host such as solveit can load it and use the safe API (searching, downloading, uploading, creating folders) by default. Renaming, moving, trash, deletion, and sharing are documented in the skill but are not enabled, so you turn them on when a task needs them.

In [ ]:
#| hide
await folder.delete()